# DEEPLRN — Google Colab Quickstart

This notebook gets the [DEEPLRN](https://github.com/Airnone/DEEPLRN-PROJECT) project running in Colab so you can use a real GPU — the project's own development machine is an Apple Silicon Mac with no CUDA, so Colab is where actual training runs should happen.

**What DEEPLRN does**: extracts neutral audit findings, entities, amounts, and relationships from Philippine Commission on Audit (COA) reports using a multi-task RoBERTa-based model. See the repo's `README.md` for the full picture.

**How to use this notebook**: run the cells top to bottom on a fresh Colab runtime. Each section below explains what its cell does and why it's there before you run it. The setup cells (clone → install → Tesseract → GPU check → Drive mount → tests) are safe to re-run if your runtime disconnects and you reconnect — they're written to skip work that's already done.

## 1. Clone the repository

This pulls down the project's source code **and** the real Pasig City COA report files used as example data — both are committed to the repo, so one `git clone` gets you everything needed to try the pipeline immediately, with no separate upload step.

Note: this clones the **`test`** branch specifically — that's where all the actual work lives; the repo's default `main` branch is still just the empty initial commit.

The check below skips re-cloning if you've already run this cell once in the current session (e.g. after a reconnect where the disk survived).

In [ ]:
import os

if os.path.basename(os.getcwd()) != "DEEPLRN-PROJECT":
    if not os.path.isdir("DEEPLRN-PROJECT"):
        !git clone -b test https://github.com/Airnone/DEEPLRN-PROJECT.git
    %cd DEEPLRN-PROJECT

print("Working directory:", os.getcwd())

## 2. Install the package

DEEPLRN is a proper installable Python package (see `pyproject.toml`). `pip install -e .` installs it in **editable** mode, meaning both `import deeplrn` and the `python -m deeplrn <subcommand>` CLI work immediately, and any edits you make to the source in this Colab session take effect without reinstalling. The `[dev]` extra pulls in `pytest` too, for the sanity check later in this notebook.

This is the same install path already verified to work locally — nothing Colab-specific here.

In [ ]:
!pip install -q -e ".[dev]"

## 3. Install Tesseract OCR

Some COA report pages are scanned images rather than native text (this is genuinely common — the project's own Pasig City sample data includes scanned pages), and DEEPLRN falls back to Tesseract OCR to read them. On the Mac this project was developed on, that meant `brew install tesseract`; **Colab runs Ubuntu**, so the equivalent is `apt-get`. Without this step, OCR fallback will fail with a "tesseract not found" error the first time it's needed.

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr

## 4. Check GPU availability

This is the actual reason to be in Colab instead of the local Mac setup: a CUDA GPU makes training dramatically faster than CPU (or even Apple's MPS backend). If this prints "No GPU detected", go to **Runtime → Change runtime type → Hardware accelerator → GPU**, then **Runtime → Restart session** and re-run from the top.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected.")
    print("Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU, then Runtime -> Restart session.")

## 5. Mount Google Drive for checkpoint persistence

This is a Colab-specific gotcha that didn't exist on the local Mac setup: **Colab's local disk is wiped every time the runtime recycles or times out**, which will happen mid-way through anything but a short run. If a training run or hyperparameter sweep writes its checkpoints to local disk only, a disconnect loses all of it.

This cell mounts your Google Drive and defines `DRIVE_OUTPUT_ROOT` — point every `--output` flag (for `deeplrn prepare`, `deeplrn split`, `deeplrn train`, and `scripts/hparam_sweep.py`) at a subfolder of this path so results survive a runtime recycle. This only works inside an actual Colab runtime — it will fail if you try to run this notebook somewhere else.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/deeplrn_runs"
os.makedirs(DRIVE_OUTPUT_ROOT, exist_ok=True)
print(f"Training outputs will be written under: {DRIVE_OUTPUT_ROOT}")

## 6. Run the test suite

A sanity check that the environment is actually healthy — package installed correctly, dependencies resolved, nothing broken by the Colab environment specifically — before trusting it with real work. The project currently has 58 tests covering extraction, chunking, schema validation, training/resume, and ablations end to end on synthetic data.

In [ ]:
!python -m pytest -q

## 7. Example: preprocess a real Pasig City file

Preprocessing means text extraction (from PDF via `pdfplumber`, or DOCX via `python-docx`) followed by sentence-aware chunking into RoBERTa-sized windows. This is the first real pipeline stage and needs no annotations or trained model — it's the easiest way to confirm the pipeline works end to end on this data.

The input file below is already present because it was committed to the repo in step 1 — no upload needed. This is the "Observations and Recommendations" section, which is where actual audit findings live in a COA report.

In [ ]:
!python -m deeplrn --input "Pasig-City-Annual-Audit-Report-2024/AAR/09-PasigCity2024_Part2-Observations_and_Recomm.docx" --output output/

## 8. Next steps: prepare → split → train → sweep

These stages need real **annotations** first — human-labeled entities, relations, and a finding category per document, matching the schema in `deeplrn/schema.py` — which don't exist yet for this sample data. The commands are left commented out below as a ready-to-use template for once annotations exist (put them in an `annotations/` folder):

1. **`deeplrn prepare`** — combines annotations with the source PDF/DOCX text to produce trainable JSON records (token IDs, BIO tags, finding label, relation candidates).
2. **`deeplrn split`** — builds a leakage-resistant train/validation/test manifest, keeping every year of a given LGU inside one partition and clustering near-duplicate reports together.
3. **`deeplrn train`** — trains a single run, or use **`scripts/hparam_sweep.py`** to run a guarded learning-rate × batch-size grid search that halts automatically on a NaN loss, a stuck 0.0 F1, or a plateaued score — and never touches the `test` split.

All three write into the Drive-backed `DRIVE_OUTPUT_ROOT` from step 5, so results survive a runtime recycle.

In [ ]:
# 1. Prepare: turn annotations + source PDFs/DOCX into trainable records
# !python -m deeplrn prepare --annotations annotations/ --pdf-root . --output /content/drive/MyDrive/deeplrn_runs/records

# 2. Split: leakage-resistant train/validation/test manifest, grouped by LGU + year
# !python -m deeplrn split --records /content/drive/MyDrive/deeplrn_runs/records --output /content/drive/MyDrive/deeplrn_runs/manifests/split.json --seed 13

# 3a. Train a single run
# !python -m deeplrn train --manifest /content/drive/MyDrive/deeplrn_runs/manifests/split.json --output /content/drive/MyDrive/deeplrn_runs/checkpoints --epochs 10

# 3b. Or run the guarded hyperparameter sweep
# !python scripts/hparam_sweep.py --manifest /content/drive/MyDrive/deeplrn_runs/manifests/split.json --output-root /content/drive/MyDrive/deeplrn_runs/checkpoints/sweep --results /content/drive/MyDrive/deeplrn_runs/sweeps/sweep_results.json